# 🎙️ Autism Emotion Recognition — Live Demo
**USAII Global AI Hackathon 2026 · Cora & Mujahid**

Speak into your mic (or upload a clip) → **our trained model** reads the emotion in your voice.

This runs the model *we trained*: wav2vec2 (superb) embeddings → LogisticRegression, trained on
RAVDESS — **64% speaker-independent accuracy** (8 emotions, chance = 12.5%).

> ⚠️ **Honest note.** It's trained on *neurotypical adult* speech (RAVDESS). On autistic kids it will
> sometimes be **confidently wrong** — that's exactly the problem we're solving. So it **abstains when
> unsure**, and the full project adds autism-specific adaptation (ASDSpeech continued-pretraining +
> per-wearer calibration). Voice is the live channel; face only calibrates.

Run the cells top to bottom.

In [ ]:
!apt-get -qq install -y ffmpeg >/dev/null 2>&1 || true
!pip -q install transformers librosa soundfile scikit-learn joblib

In [ ]:
import torch, joblib, urllib.request
from transformers import AutoFeatureExtractor, AutoModel

# our trained classifier (69 KB) from the repo
urllib.request.urlretrieve(
    "https://github.com/xqscora/usaii-autism-emotion/raw/master/ravdess_w2v_ser_model.joblib",
    "model.joblib")
B = joblib.load("model.joblib")

BACKBONE = "superb/wav2vec2-base-superb-er"
fe = AutoFeatureExtractor.from_pretrained(BACKBONE)
bb = AutoModel.from_pretrained(BACKBONE).eval()
print("loaded. emotions:", B["emotions"])

In [ ]:
def predict_emotion(speech, sr=16000, conf_threshold=0.40):
    inp = fe(speech, sampling_rate=sr, return_tensors="pt")
    with torch.no_grad():
        emb = bb(**inp).last_hidden_state.mean(dim=1).numpy()
    emb = B["scaler"].transform(emb)
    probs = B["model"].predict_proba(emb)[0]
    classes = B["model"].classes_
    all_probs = {B["emotions"][int(c)]: round(float(p), 3) for c, p in zip(classes, probs)}
    conf = float(probs.max())
    top = B["emotions"][int(classes[int(probs.argmax())])]
    label = top if conf >= conf_threshold else "uncertain (abstain)"
    return label, conf, all_probs

## 🎤 Option A — speak into your mic
Run this, allow mic access, talk for 4 seconds.

In [ ]:
from IPython.display import Javascript, display
from google.colab import output
import base64, librosa, tempfile, os

_REC = '''
async function record(ms){
  const stream = await navigator.mediaDevices.getUserMedia({audio:true});
  const mime = MediaRecorder.isTypeSupported('audio/webm;codecs=opus')
    ? 'audio/webm;codecs=opus' : 'audio/webm';
  const rec = new MediaRecorder(stream, {mimeType: mime});
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(r => rec.onstop = r);
  rec.start(); await new Promise(r => setTimeout(r, ms)); rec.stop(); await stopped;
  stream.getTracks().forEach(t => t.stop());
  const buf = await (new Blob(chunks, {type: mime})).arrayBuffer();
  const bytes = new Uint8Array(buf); let bin = '';
  for (let i=0;i<bytes.length;i++) bin += String.fromCharCode(bytes[i]);
  return btoa(bin);
}
'''

def _decode_mic_bytes(raw: bytes, sr=16000):
    """MediaRecorder sends webm — librosa/soundfile need a file extension + ffmpeg."""
    fd, path = tempfile.mkstemp(suffix='.webm')
    try:
        os.write(fd, raw)
        os.close(fd)
        speech, _ = librosa.load(path, sr=sr)
        return speech
    finally:
        if os.path.exists(path):
            os.unlink(path)

def record(sec=4):
    display(Javascript(_REC))
    print(f"Recording {sec}s -- speak now!")
    b64 = output.eval_js(f'record({sec*1000})')
    return _decode_mic_bytes(base64.b64decode(b64))

speech = record(4)
label, conf, all_probs = predict_emotion(speech)
print(f"\n🎭 emotion: {label}   (confidence {conf:.2f})")
print("all probs:", all_probs)

## 📁 Option B — upload an audio file
If the mic cell is fussy, upload a .wav/.mp3 of someone talking.

In [ ]:
from google.colab import files
import librosa
up = files.upload()
speech, _ = librosa.load(list(up.keys())[0], sr=16000)
label, conf, all_probs = predict_emotion(speech)
print(f"🎭 emotion: {label}   (confidence {conf:.2f})")
print("all probs:", all_probs)

## 🔭 What's next (the actual project)
This baseline (64% on neurotypical RAVDESS) proves the pipeline. The real work is making it right for autistic kids:
1. **Continued pretraining on ASDSpeech** (197 autistic children's voices) so the backbone isn't neurotypical-only.
2. **Per-wearer calibration** — a few samples per child, not one generic template.
3. **Uncertainty-aware abstention** so a confident wrong read never gets shown.
4. **Face as calibration** (FER-Autism), not a live camera.

Repo: https://github.com/xqscora/usaii-autism-emotion

## 🎬 Option C — video with audio + face (multimodal)

Upload a **video clip with sound** (mp4). We read **audio emotion** (wav2vec2) and **face emotion** (FER-Autism ResNet18, 66% on test) and **fuse** them.

When voice and face disagree (e.g. laugh sounds like fear), fusion is often more stable than either alone.

In [ ]:
!pip -q install torch torchvision opencv-python-headless
import urllib.request, torch, cv2, librosa, numpy as np
from pathlib import Path

# face model (~45MB) from repo
urllib.request.urlretrieve(
    "https://github.com/xqscora/usaii-autism-emotion/raw/master/fer_autism_model.pt",
    "fer_autism_model.pt")

# clone inference helpers (or paste minimal fusion below)
!git clone -q --depth 1 https://github.com/xqscora/usaii-autism-emotion.git _repo
import sys
sys.path.insert(0, "_repo/code")
from demo_multimodal import process_video

from google.colab import files
print("Upload a short mp4 with audio (5-15s works best)")
up = files.upload()
vid = list(up.keys())[0]
process_video(Path(vid), w_audio=0.45, w_face=0.55, out_path=Path("labeled_multimodal.mp4"), show=False, face_weights=Path("fer_autism_model.pt"))
print("done. download labeled_multimodal.mp4 from Files panel")